# **KKBOX Churn Prediction And Retention Intelligence System - Initial Data & Business Understanding Exploration**

---

## **1. Objective**

This notebook explores the KKBOX dataset to build an initial understanding of customer churn.

The goal is to:
- Understand the structure and quality of the data
- Identify potential issues and limitations
- Explore early patterns related to churn
- Generate initial business hypotheses

At this stage, the focus is exploratory. No modeling or heavy preprocessing is applied yet.


---

## **2. Setup**

In [51]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(PROJECT_ROOT)

pd.set_option("display.max_columns", None)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

target = "is_churn"

---

## **3. Data Overview**

In [ ]:
# 3.1 Train Dataset (Target)
train = pd.read_csv("data/raw/train_v2.csv")

train.info()
train.head()
print("\nMissing values in train:")
print(train.isnull().sum())
print("\nDuplicates users in train:")
print("Duplicated users:", train["msno"].duplicated().sum())
sns.countplot(x=target, data=train)
plt.title("Churn Distribution")
plt.show()

print(f"Churn rate: {train[target].mean()*100:.2f}%")


**Insight**
- The dataset is clean (no missing values, no duplicates)
- Churn rate is ~9%, indicating class imbalance
- This will be important for model evaluation later

In [ ]:
# 3.2 Members Dataset
members = pd.read_csv("data/raw/members_v3.csv")

members.info()
members.head()
print("\nMissing values in members:")
print(members.isnull().mean().round(2))
print("\nDuplicated customer IDs:", members["msno"].duplicated().sum())
print("Duplicated rows:", members.duplicated().sum())
sns.histplot(members["bd"], bins=30)
plt.title("Age Distribution")
plt.show()

**Insight**
- Age (`bd`) contains invalid values (0, negative, extreme)
- Gender has a high percentage of missing values (~65%)
- The dataset has no duplicates
- Some numerical variables behave like categorical (city, registered_via)

In [ ]:
# 3.3 Transactions Dataset
transactions = pd.read_csv("data/raw/transactions_v2.csv")

transactions.info()
transactions.head()
print("\nMissing values in transactions:")
print(transactions.isnull().mean().round(2))
sns.countplot(x="is_cancel", data=transactions)
plt.title("Cancellation")
plt.show()

sns.countplot(x="is_auto_renew", data=transactions)
plt.title("Auto Renew")
plt.show()

**Insight**
- Multiple transactions per user
- Auto-renew and cancellation stand out as key behavioral signals
- Payment variables show high variability

In [ ]:
# 3.4 User Logs Dataset
user_logs = pd.read_csv("data/raw/user_logs_v2.csv")

user_logs.info()
print("\nMissing values in user logs:")
print(user_logs.isnull().mean().round(2))
sns.histplot(user_logs["total_secs"], bins=30)
plt.title("Listening Time")
plt.show()

sns.histplot(user_logs["total_secs"], bins=30, log_scale=True)
plt.title("Listening Time (Log Scale)")
plt.show()

 **Insight**
- Very large dataset (multiple rows per user)
- Engagement is highly skewed (few heavy users)

---

## **4. Quick Merge (Initial Exploration)**

In [ ]:
train_members = train.merge(members, on="msno", how="left")
sns.boxplot(x="is_churn", y="bd", data=train_members)
plt.title("Age vs Churn")
plt.show()

 **Insight**
- Demographics alone do not clearly explain churn
- Behavioral data will likely be more informative

---

## **5. Build Customer-Level Dataset**

In [ ]:
# 5.1 Aggregate User Logs
user_logs = pd.read_csv("data/raw/user_logs_v2.csv")

user_logs_agg = user_logs.groupby("msno").agg({
    "total_secs": "sum",
    "num_unq": "sum"
}).reset_index()
print("\nAggregated user logs info:")
print(user_logs_agg.info())

In [ ]:
# 5.2 Aggregate Transactions

# Convert date columns
transactions["transaction_date"] = pd.to_datetime(
    transactions["transaction_date"],
    errors="coerce"
)

transactions["membership_expire_date"] = pd.to_datetime(
    transactions["membership_expire_date"],
    errors="coerce"
)

# Aggregate transaction data at customer level
transactions_agg = transactions.groupby("msno").agg({
    "actual_amount_paid": ["sum", "mean"],
    "payment_plan_days": ["sum", "mean"],
    "plan_list_price": "mean",
    "is_auto_renew": ["mean", "max"],
    "is_cancel": ["mean", "max"],
    "transaction_date": ["count", "min", "max"],
    "membership_expire_date": "max"
}).reset_index()

transactions_agg.columns = [
    "msno",
    "total_amount_paid",
    "avg_amount_paid",
    "total_payment_plan_days",
    "avg_payment_plan_days",
    "avg_plan_list_price",
    "auto_renew_rate",
    "has_auto_renew",
    "cancel_rate",
    "has_cancelled",
    "transaction_count",
    "first_transaction_date",
    "last_transaction_date",
    "membership_expire_date"
]

transactions_agg.head()


print(transactions["transaction_date"].head())
print(transactions["transaction_date"].dtype)


In [ ]:
# 5.3 Final Dataset

df.info()
df = train.merge(members, on="msno", how="left")
df = df.merge(transactions_agg, on="msno", how="left")
df = df.merge(user_logs_agg, on="msno", how="left")



---

## **6. Key Business Questions**

In [ ]:
# 6.1 Which segments churn more?
# Churn rate by auto-renew
auto_renew_churn = df.groupby("has_auto_renew")[target].mean() * 100

print("Churn Rate by Auto-Renew Status (%):")
for status, value in auto_renew_churn.items():
   print(f"Auto-Renew = {int(status)} → {value:.2f}%")

# Churn rate by cancellation
cancel_churn = df.groupby("has_cancelled")[target].mean() * 100

print("\nChurn Rate by Cancellation Status (%):")
for status, value in cancel_churn.items():
   print(f"Has Cancelled = {int(status)} → {value:.2f}%")


**Insight**
- Cancellation is the strongest churn signal
- Auto-renew drastically reduces churn risk


In [ ]:
# 6.2 Behavioral patterns
sns.boxplot(data=df, x="is_churn", y="total_secs")
plt.title("Listening vs Churn")
plt.show()
df.groupby("is_churn")[["total_secs", "num_unq"]].mean()

 **Insight**
- Lower engagement is associated with higher churn
- Churned users listen less and explore less content

In [ ]:
# 6.3 Payment behavior
sns.boxplot(data=df, x="is_churn", y="avg_amount_paid")
plt.show()

df.groupby("is_churn")["avg_amount_paid"].mean()


 **Insight**
- Payment-related variables show strong differences
- Suggests churn may be linked to plan type or pricing


In [ ]:
# 6.4 Timing of churn
df["registration_init_time"] = pd.to_datetime(
    df["registration_init_time"].astype("Int64").astype(str),
    format="%Y%m%d",
    errors="coerce"
)

df["customer_tenure"] = (
    df["last_transaction_date"] - df["registration_init_time"]
).dt.days
sns.boxplot(data=df, x="is_churn", y="customer_tenure")
plt.show()



**Insight**
- Churn appears related to lifecycle stage
- Early-stage users may behave differently from long-term users


In [ ]:
# 6.5 Early risk signals
df["listening_group"] = pd.qcut(
    df["total_secs"], q=4,
    labels=["Low", "Medium-Low", "Medium-High", "High"]
)

df.groupby("listening_group")[target].mean() * 100


**Insight**
- Low engagement group shows highest churn
- Could be used for early intervention strategies


In [ ]:
# 6.6 Risk segmentation

risk_table = df.groupby(
["has_auto_renew", "has_cancelled"]
)[target].agg(["mean", "count"]).reset_index()
risk_table["mean"] = risk_table["mean"] * 100
risk_table.columns = ["auto_renew", "cancelled", "churn_rate", "customers"]
risk_table = risk_table.sort_values("churn_rate", ascending=False).reset_index(drop=True)
risk_labels = ["High", "Medium", "Low"]
risk_table["risk_level"] = risk_labels[:len(risk_table)]
risk_table = risk_table[["risk_level", "auto_renew", "cancelled", "churn_rate", "customers"]]
risk_table["churn_rate"] = risk_table["churn_rate"].round(2)
print(risk_table)


**Insight**
- Highest risk group: customers who have cancelled their subscription, regardless of auto-renew status (60.26%)

- Medium risk: customers without auto-renew and no cancellation, indicating low commitment (30.26%)

- Lowest risk: customers with auto-renew and no cancellation, showing strong retention (1.70%)

- Key insight: cancellation behavior is the strongest predictor of churn, outweighing the effect of auto-renew.


In [ ]:
# 7. ARPU only for churned customers
churned_customers = df["is_churn"].sum()
arpu_churn = df[df["is_churn"] == 1]["avg_amount_paid"].mean()
revenue_lost = churned_customers * arpu_churn

print(f"ARPU (churned users): €{arpu_churn:.2f}")
print(f"Estimated revenue lost: €{int(revenue_lost):,}")



**Insight**
- ARPU (churned users): €367.03
- Estimated revenue lost: €32,052,714

In [ ]:
# 8. ARPU only for no churned customers
arpu_non_churn = df[(df["is_churn"] == 0) & (df["avg_amount_paid"] > 0)]["avg_amount_paid"].median()

print(f"ARPU (non-churn users): €{arpu_non_churn:.2f}")


**Insight**
- ARPU (non-churn users): €149.00

In [ ]:
# 9. Revenue opportunity estimation (simple proxy)
medium_risk_customers = 80777
potential_saved = medium_risk_customers * 0.10 # reduce churn by 10%

print(f"Potential customers saved whitin the medium-risk segment: {int(potential_saved)}")

**Insight**
- Potential customers saved whitin the medium-risk segment: 8077 customers

In [ ]:
# 10. Estimate revenue impact using real ARPU
revenue_saved = potential_saved * arpu

print(f"Potential revenue saved: €{int(revenue_saved):,}")


**Insight**
- Potential revenue saved within the medium-risk churn segment: €1,203,577

---

####################################################################################################################################################

---

## **7. Conclusion**

The exploratory analysis indicates that churn in our platform is primarily associated with **behavioral factors** rather than demographic characteristics.

### Key Signals Identified

- **Subscription behavior**, especially cancellation and auto-renewal status  
- **Payment-related patterns**, such as plan duration and average amount paid  
- **User engagement**, particularly listening activity and content exploration  

These findings suggest that churn should be understood as a **behavioral process**, where declining engagement and cancellation intent play a central role.

---

### Data Perspective

Several important considerations were identified:

- The dataset is generally clean in key tables (e.g., `train`), with no missing values or duplicated user IDs  
- However, some variables require careful treatment:
  - `bd` (age) contains unrealistic values (zeros, negatives, extreme values)  
  - `gender` has a high proportion of missing values (~65%)  
- Some numerical variables behave as categorical (e.g., `city`, `registered_via`) and should be handled accordingly in future modeling  
- Data is distributed across multiple tables with different granularities (user-level vs. transaction-level vs. activity logs), requiring aggregation and careful feature engineering  

---

### Business Perspective

Churn is not only a customer retention issue but also a **significant revenue risk**:

- Churned users show higher ARPU (€367.03) compared to retained users (€149.00), indicating a disproportionate loss of higher-value customers  
- Under a simplified retention scenario, retaining 10% of customers in the medium-risk segment could recover approximately €1.2 million in revenue  

---

### Recommended Actions

- Incentives to encourage auto-renew activation  
- Re-engagement campaigns for low-activity users  
- Monitoring cancellation behavior as an early intervention trigger  

---

### Next Step

These insights provide a strong foundation for the next phase of the project, where **feature engineering and predictive modeling** will focus on capturing these behavioral patterns while addressing the identified data limitations.